# Content Embeddings — Weighted Tag Vectors & Cross-Domain Recommendations

Builds a content-based similarity signal from AniList genre/tag data, used for two purposes:
- **Cold-start handling** — recommending anime/manga with little or no rating history, based on what they're about rather than who liked them
- **Cross-domain bridging** — recommending manga based on an anime a user liked (and vice versa), without needing shared user identities across the two ratings datasets (which don't exist across platforms — see `01_data_collection.ipynb`)

**Approach note:** an earlier version of this step used sentence-transformer embeddings on plot synopses (and later, synopsis + genre/tag text combined) to measure similarity. That approach under-performed on real test cases — e.g. it rated *Mob Psycho 100* as more similar to *Attack on Titan* than *Fullmetal Alchemist: Brotherhood* was, which doesn't match genre-savvy fan intuition. The issue: raw synopsis text captures plot events, not tone/theme, and even tag-enriched text over-weighted common, low-signal tags ("Male Protagonist", "Shounen") while diluting rare, meaningful ones ("Cannibalism", "Steampunk") on titles with long tag lists. The approach below instead builds explicit **rarity-weighted tag vectors** (an IDF-style scheme), which directly fixes both problems — validated below against known-similar and known-different anime pairs.

In [1]:
import json
import os
import math
from collections import Counter

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz

## Load anime & manga metadata

Both datasets come from `01_data_collection.ipynb`'s AniList pull — same tag vocabulary, which is what makes cross-domain comparison valid later.

In [2]:
DATA_DIR = os.path.join("..", "data")

with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
    anime_content = [json.loads(line) for line in f]

with open(os.path.join(DATA_DIR, "manga_data.jsonl"), "r") as f:
    manga_content = [json.loads(line) for line in f]

print(len(anime_content), len(manga_content))

5000 5000


## Build rarity-weighted tag vectors

Each anime/manga is represented as a vector over the *combined* anime+manga tag vocabulary, where each tag's weight is scaled by how rare it is (an IDF-style scheme: `log(total_items / (times_this_tag_appears + 1))`).

**Why rarity weighting, not raw tag presence:** a tag like "Male Protagonist" appears on a huge fraction of all anime and tells you almost nothing when shared between two titles. A tag like "Cannibalism" or "Steampunk" is rare and genuinely indicates deep similarity when shared. Raw (unweighted) tag overlap treats both the same, and also gets diluted on titles with unusually long tag lists (more total tags to add unrelated noise). Rarity weighting fixes both issues at once.

**Why the vocabulary/weights are computed across anime + manga together, not separately:** for cross-domain similarity to mean anything, "Tragedy" needs to carry the same weight whether it's an anime's tag or a manga's tag — computing weights separately per domain would make the two vector spaces incomparable.

In [3]:
tag_counts = Counter()
for item in anime_content + manga_content:
    for t in item.get('tags', []):
        tag_counts[t['name']] += 1

total_items = len(anime_content) + len(manga_content)

def tag_weight(tag_name):
    # Rare tags get a high weight, common tags get pushed toward ~0
    return math.log(total_items / (tag_counts[tag_name] + 1))

all_tags = sorted(tag_counts.keys())
tag_to_col = {t: i for i, t in enumerate(all_tags)}

def build_tag_vector(item):
    vec = np.zeros(len(all_tags))
    for t in item.get('tags', []):
        vec[tag_to_col[t['name']]] = tag_weight(t['name'])
    return vec

anime_tag_vectors = np.array([build_tag_vector(a) for a in anime_content])
manga_tag_vectors = np.array([build_tag_vector(m) for m in manga_content])

print(anime_tag_vectors.shape, manga_tag_vectors.shape)

np.save(os.path.join(DATA_DIR, "anime_tag_vectors.npy"), anime_tag_vectors)
np.save(os.path.join(DATA_DIR, "manga_tag_vectors.npy"), manga_tag_vectors)

(5000, 422) (5000, 422)


## Validate against known anime pairs

Sanity check before trusting this at scale: compare a few anime pairs with well-known ground-truth similarity (from genre-savvy fan intuition), and confirm the *ranking* matches — not just that the numbers look plausible in isolation.

In [4]:
def find_anime_index(title_fragment):
    """Look up an anime's index by a fragment of its romaji or english title."""
    for i, a in enumerate(anime_content):
        romaji = (a['title']['romaji'] or '').lower()
        english = (a['title'].get('english') or '').lower()
        if title_fragment.lower() in romaji or title_fragment.lower() in english:
            return i
    return None

idx_aot = find_anime_index("Shingeki no Kyojin")     # Attack on Titan
idx_fmab = find_anime_index("FULLMETAL ALCHEMIST")   # Fullmetal Alchemist: Brotherhood
idx_hxh = find_anime_index("HUNTER")                 # Hunter x Hunter
idx_op = find_anime_index("ONE PIECE")               # One Piece
idx_mob = find_anime_index("Mob Psycho 100 II")      # Mob Psycho 100 II

In [5]:
def tag_similarity(i, j, vectors_a, vectors_b):
    return cosine_similarity([vectors_a[i]], [vectors_b[j]])[0][0]

print("AOT vs FMAB:       ", tag_similarity(idx_aot, idx_fmab, anime_tag_vectors, anime_tag_vectors))
print("AOT vs Mob Psycho: ", tag_similarity(idx_aot, idx_mob, anime_tag_vectors, anime_tag_vectors))
print("HxH vs One Piece:  ", tag_similarity(idx_hxh, idx_op, anime_tag_vectors, anime_tag_vectors))

# Expected: AOT-FMAB clearly > AOT-Mob Psycho — deep thematic overlap (Military,
# Tragedy, Gore, Steampunk, Cannibalism) should beat shallow shared demographic
# tags alone ("Shounen", "Male Protagonist")

AOT vs FMAB:        0.31745438007808424
AOT vs Mob Psycho:  0.11173893531864643
HxH vs One Piece:   0.12337914662139624


## Cross-domain recommendation: anime → manga

Given an anime, finds the most similar manga by tag-vector cosine similarity — the actual cross-domain bridge described in the intro. Same-franchise matches (e.g. an anime's own source manga, or its direct spin-offs) are filtered out via fuzzy title matching, since the goal is genuine discovery, not surfacing the obvious "here's the manga this anime is based on" result at the top.

In [6]:
def is_same_franchise(title_a, title_b, threshold=60):
    """Fuzzy title match — catches variants like 'Attack on Titan' vs.
    'Attack on Titan: Before the Fall', not just exact duplicates."""
    romaji_score = fuzz.partial_ratio(title_a['romaji'] or '', title_b['romaji'] or '')
    english_score = fuzz.partial_ratio(title_a.get('english') or '', title_b.get('english') or '')
    return max(romaji_score, english_score) >= threshold

In [7]:
def recommend_manga_for_anime(anime_idx, k=10, exclude_same_franchise=True):
    anime_vec = anime_tag_vectors[anime_idx].reshape(1, -1)
    similarities = cosine_similarity(anime_vec, manga_tag_vectors)[0]

    source_title = anime_content[anime_idx]['title']

    # Pull more candidates than needed, since franchise matches get filtered out
    candidate_indices = similarities.argsort()[::-1][:k * 5]

    results = []
    for i in candidate_indices:
        candidate_title = manga_content[i]['title']
        if exclude_same_franchise and is_same_franchise(source_title, candidate_title):
            continue
        results.append((candidate_title, similarities[i]))
        if len(results) == k:
            break

    return results

## Demo: manga recommendations for Attack on Titan

In [14]:
print(anime_content[12]['title']['english'])
recommend_manga_for_anime(12, k=10)

Sword Art Online


[({'romaji': 'THE NEW GATE', 'english': 'The New Gate'},
  np.float64(0.3719999723166487)),
 ({'romaji': 'Death Game Manga no Kuromaku Satsujinki no Imouto ni Tensei shite Shippai shita',
   'english': None},
  np.float64(0.34361160502144006)),
 ({'romaji': 'Nakanohito Genome [Jikkyouchuu]', 'english': None},
  np.float64(0.33947821536725403)),
 ({'romaji': 'Shangri-La Frontier: Kusoge Hunter, Kami ge ni Idoman to su',
   'english': 'Shangri-La Frontier'},
  np.float64(0.31014536697647455)),
 ({'romaji': 'Übel Blatt', 'english': 'Ubel Blatt'},
  np.float64(0.2936110256852514)),
 ({'romaji': 'Chao Shen Jixieshi', 'english': 'The Legendary Mechanic'},
  np.float64(0.29289189405368726)),
 ({'romaji': 'Kawaikereba Hentai demo Suki ni Natte Kuremasu ka?',
   'english': None},
  np.float64(0.28669117261929916)),
 ({'romaji': 'Reader: Ingneunja', 'english': None},
  np.float64(0.28083995727759914)),
 ({'romaji': 'Itai no wa Iya nano de Bougyoryoku ni Kyokufuri Shitai to Omoimasu.',
   'englis